# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields by @id

from pprint import pprint

record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s) in the dataset.")

for record_set in record_sets:
    print(f"\nRecordSet: {record_set.name} (@id: {record_set.id})")
    print("  Fields (by @id):")
    for field in record_set.fields:
        print(f"   - {field.name} (@id: {field.id})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, we load data from all available record sets

dataframes = {}
for record_set in record_sets:
    print(f"Loading records from RecordSet: {record_set.name} (@id: {record_set.id})")
    records = list(dataset.records(record_set=record_set.id))
    if len(records) > 0:
        # Only create dataframe if records exist
        df = pd.DataFrame(records)
        dataframes[record_set.id] = df
        print(f"RecordSet '{record_set.name}' ({record_set.id}) has {df.shape[0]} records, columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print("No records found.")

if len(dataframes) == 0:
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For demonstration, select the main record set containing survey responses
# If available, choose a numeric field for analysis (e.g., PHQ-9 or GAD-7 score)

# List existing record set IDs
print("Available record set IDs:")
for idx, rset_id in enumerate(dataframes.keys()):
    print(f"{idx}. {rset_id}")

# Set your record_set_id (choose first available as example)
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]

    print(f"Columns in record set '{record_set_id}': {df.columns.tolist()}")

    # Example: Use PHQ-9 score or GAD-7 score if present
    candidate_fields = [col for col in df.columns if any(x in col.lower() for x in ['phq', 'gad', 'psq', 'score'])]
    if candidate_fields:
        numeric_field = candidate_fields[0]
    else:
        # fallback to first numeric column
        numeric_field = df.select_dtypes('number').columns[0]
    print(f"Using numeric field for analysis: {numeric_field}")
    # Handle missing values for demonstration
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize numeric field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, norm_col]].head())

    # Attempt grouping by a common demographic field like 'gender' or 'level_of_education' if present
    group_field = None
    for col in df.columns:
        if col.lower() in ['gender', 'sex', 'level_of_education', 'education']:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df)
    else:
        print("No suitable categorical field found for grouping.")
else:
    print('No suitable data available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    # Plot the distribution of the numeric field
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group_field is available, show boxplot
    if group_field:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we loaded a Croissant-formatted survey dataset from Kilifi County, Kenya, using the `mlcroissant` Python library.*

- We discovered available record sets and fields using their `@id`s for robust referencing.
- We extracted records from main tables and loaded them into pandas DataFrames for tabular manipulation.
- Exploratory steps included basic filtering and normalization of a chosen numeric mental health score field, and groupwise summary if an appropriate demographic field was available.
- Visualizations gave insight into numeric distributions and group differences.

**Next steps:** Depending on research interests, more advanced analyses (e.g. regression, missing value imputation, or machine learning) can be performed leveraging the clearly defined schema and data provenance enabled by Croissant.
